## Deliverable 2
Markus Valland, Jørgen Skarra, Wiktor Correia-Myrhagen



### Task 1

In [1]:

import re
import json
from pathlib import Path

import numpy as np
import pandas as pd


#Task 1 File ingestion and regular expressions
def read_file(path) -> tuple[str, str]:
    raw = Path(path).read_bytes()
    try:
        return raw.decode("utf-8"), "utf-8"
    except UnicodeDecodeError:
        return raw.decode("iso-8859-1"), "iso-8859-1"

#Compile regex pattern for log line parsing 
LOG_PATTERN = re.compile(
    r"^\[(?P<timestamp>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})\]\s+"
    r"\[(?P<register_id>[^\]]+)\]\s+"
    r"item=(?P<item>.+?)\s+"
    r"price=(?P<price>\d+(?:\.\d+)?)\s+"
    r"(?P<currency>[A-Z]{3})\s+"
    r"\((?P<status>SUCCESS|REFUND|CANCELLED)\)$"
)

#Function to parse log lines into structured records
def parse_log(text: str) -> list[dict]:
    records = []
    for line in text.splitlines():
        match = LOG_PATTERN.match(line.strip().lstrip("\ufeff"))
        if match:
            record = match.groupdict()
            record["price"] = float(record["price"])
            records.append(record)
    return records

### task 2

In [2]:
def find_log_files(directory: str | Path, extension: str = ".log") -> list[Path]:
    #Finds all the logfiles
    directory = Path(directory) # converts
    log_files = [] 
    for item in directory.iterdir():
        if item.is_dir():
            log_files.extend(find_log_files(item, extension))  # recursive call, looks into the subfolders
        elif item.is_file() and item.suffix == extension: #only includes files ending .log
            log_files.append(item)
    return log_files

In [3]:
log_files = find_log_files("data")
print(f"Found {len(log_files)} log files")
for f in log_files:
    print(f) #print log files

Found 9 log files
data/2026/science_building/POS_01/register_sales.log
data/2026/science_building/POS_03/register_sales.log
data/2026/science_building/POS_02/register_sales.log
data/2026/student_union/POS_01/register_sales.log
data/2026/student_union/POS_03/register_sales.log
data/2026/student_union/POS_02/register_sales.log
data/2026/library_cafe/POS_01/register_sales.log
data/2026/library_cafe/POS_03/register_sales.log
data/2026/library_cafe/POS_02/register_sales.log


In [4]:
def find_active_registers(data) -> set[str]:
    """Looks through JSON file and collect ID of active registers."""
    active = set()
    if isinstance(data, dict): #looks into dictonaries
        if data.get("active") is True and "id" in data:
            active.add(data["id"])
        for value in data.values():
            active |= find_active_registers(value)   # recursive call
    elif isinstance(data, list): # looks into lists
        for element in data:
            active |= find_active_registers(element)  # recursive call
    return active

In [5]:
config = json.loads(Path("data/store_config.json").read_text(encoding="utf-8"))
active_registers = find_active_registers(config)
print(sorted(active_registers))
#finds active registers in store_config.json and writes them out sorted

['POS_01', 'POS_02', 'POS_03']


## Task 3: Cleaning and analysing data using NumPy


In [6]:
def remove_price_outliers(prices) -> np.ndarray: #remove outliers
    """Replaces values more than 3 standard deviations from the mean
    with the median of the valid prices. Returns a cleaned 1D array."""

    prices = np.array(prices, dtype=float).ravel()

    mean = prices.mean()
    std = prices.std()

    # Outliers deviate from the mean by more than 3 standard deviations
    is_outlier = np.abs(prices - mean) > 3 * std

    # The median is calculated from the valid prices only
    median = np.median(prices[~is_outlier])

    prices[is_outlier] = median
    return prices

In [7]:
#test file
test_prices = [4.5, 3.0, 4.2, 3.5, 3.8, 7.5, 4.5, 3.0, 4.2, 3.5, 3.8, 7.5, 999.0]
cleaned = remove_price_outliers(test_prices)
print("Before:", test_prices)
print("After: ", cleaned)

Before: [4.5, 3.0, 4.2, 3.5, 3.8, 7.5, 4.5, 3.0, 4.2, 3.5, 3.8, 7.5, 999.0]
After:  [4.5 3.  4.2 3.5 3.8 7.5 4.5 3.  4.2 3.5 3.8 7.5 4. ]


In [8]:
def compute_refund_ratio(sales_count: np.ndarray, refund_count: np.ndarray) -> np.ndarray:
    """Computes the element-wise refund ratio using NumPy array operations."""
    epsilon = 1e-6  # avoids division by zero when sales_count is 0
    sales_count = np.asarray(sales_count, dtype=float)
    refund_count = np.asarray(refund_count, dtype=float)
    return refund_count / (sales_count + epsilon)

In [9]:
sales = np.array([10, 5, 0])
refunds = np.array([2, 1, 0])
print(compute_refund_ratio(sales, refunds))

[0.19999998 0.19999996 0.        ]


## Discussion

## AI-Summary

Task 1
	Provided a regex pattern (LOG_PATTERN) with named capture groups to extract the 6 required log fields and filter out system noise. 
	Verified and approved the custom read_file function you had already created for handling dual encodings (UTF-8 and ISO-8859-1). 
	Confirmed the addition of .lstrip("\ufeff") to handle Byte Order Mark (BOM) characters during log parsing
Task 3
	Assisted in implementing remove_price_outliers to convert prices to a 1D NumPy float array, detect hardware glitch outliers deviating by >3 standard deviations, and replace them with the median of valid prices. 
	Formulated the vectorised NumPy function compute_refund_ratio using ϵ=10^(-6) to safely calculate element-wise refund ratios without division-by-zero errors. 
	Verified that both NumPy functions strictly adhere to the mathematical formulas and requirements specified in Task 3.

## Github Link
